# Experiment 1 — Manuscript & Supplemental Figures

Generates all figures for Experiment 1 (single-trial reality monitoring).

**Outputs**
- `reports/figures/manuscript/Fig3_exp1_key_results.pdf/.png` — 4-panel manuscript figure
- `reports/figures/supplemental/FigS1_exp1_confidence.pdf/.png` — confidence distributions
- `reports/figures/supplemental/FigS2_exp1_relatedness.pdf/.png` — relatedness ratings
- `reports/figures/supplemental/FigS3_exp1_gamma_z.pdf/.png` — gamma z-score detail

**Run with project venv:** `.venv/bin/jupyter nbconvert --to notebook --execute notebooks/05_figures/exp1_manuscript_figures.ipynb`

All statistics come from the trial-level data; no model re-fitting required.

In [1]:
# ── 0. Imports & config ───────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from pathlib import Path

import rmllm
from rmllm import gamma as gamma_mod

PROJ = Path(rmllm.config.PROJ_ROOT)
DATA = PROJ / 'data' / 'processed'
MAN  = PROJ / 'reports' / 'figures' / 'manuscript'
SUP  = PROJ / 'reports' / 'figures' / 'supplemental'
MAN.mkdir(parents=True, exist_ok=True)
SUP.mkdir(parents=True, exist_ok=True)

DPI_MANUSCRIPT = 700
DPI_PREVIEW    = 150

# ── Shared aesthetics ─────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'axes.labelweight': 'bold',
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.linewidth': 1.0, 'axes.facecolor': 'white',
    'figure.facecolor': 'white', 'axes.grid': False,
    'xtick.bottom': True, 'ytick.left': True,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.major.size': 4, 'ytick.major.size': 4,
    'legend.fontsize': 9, 'legend.framealpha': 0.9,
    'legend.edgecolor': '#cccccc',
})

MODEL_ORDER  = ['Gemma3:12b','Gemma3:12b-QAT','Gemma3:27b',
                'Gemma3:27b-QAT','Llama3.3:70b','Llama4:16x17b']
MODEL_LABELS = ['G3:12b','G3:12b\nQAT','G3:27b',
                'G3:27b\nQAT','L3.3:70b','L4:16x17b']

MEM_PAL    = {'SingleTurn': '#4C72B0', 'TrialChain': '#DD8452'}
MEM_LABELS = {'SingleTurn': 'Single-Turn', 'TrialChain': 'Trial-Chain'}
SRC_PAL    = {'perceived': '#2166ac',  'imagined':  '#d6604d'}
SRC_HATCH  = {'perceived': '',         'imagined':  '///'}
SRC_LABELS = {'imagined': 'Internal',  'perceived': 'External'}

def _panel_tag(ax, letter, title=''):
    ax.text(-0.13, 1.07, letter, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
    if title:
        ax.set_title(title, pad=6, fontsize=12, fontweight='bold')

print('Setup complete.')

2026-06-19 11:01:15.378 | INFO     | rmllm.config:<module>:11 - PROJ_ROOT path is: <PROJECT_ROOT>


Setup complete.


In [2]:
# ── 1. Load & preprocess data ────────────────────────────────────────────
df = pd.read_csv(DATA / 'exp1_trial_data.csv')

# Normalise column names for downstream use
df['source']   = df['source'].astype(str).str.strip().str.lower()
df['memory']   = df['memory'].astype(str).str.strip()
df['model']    = df['model'].astype(str).str.strip()
df['accuracy'] = pd.to_numeric(df['accuracy'], errors='coerce')
df['reading_hallucination'] = pd.to_numeric(df['reading_hallucination'], errors='coerce')
df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')
df['rating']     = pd.to_numeric(df['rating'],     errors='coerce')

# Perceived-only subset (for RH and accuracy analyses)
df_per = df[df['source'] == 'perceived'].copy()

print(f'Total trials: {len(df):,}  |  Perceived: {len(df_per):,}')
print('Models:', sorted(df['model'].unique()))
print('Memory:', sorted(df['memory'].unique()))
print('Source:', sorted(df['source'].unique()))

Total trials: 3,456  |  Perceived: 1,728
Models: ['Gemma3:12b', 'Gemma3:12b-QAT', 'Gemma3:27b', 'Gemma3:27b-QAT', 'Llama3.3:70b', 'Llama4:16x17b']
Memory: ['SingleTurn', 'TrialChain']
Source: ['imagined', 'perceived']


In [3]:
# ── 2. Compute panel-level summaries ──────────────────────────────────────

# Panels A & B: accuracy by model × source, split by memory
acc = (df.groupby(['model','memory','source'], observed=True)['accuracy']
         .agg(['mean','sem']).reset_index())
acc.columns = ['model','memory','source','acc_mean','acc_sem']

# Panel C: reading hallucination (perceived only) by model × memory
rh = (df_per.groupby(['model','memory'], observed=True)['reading_hallucination']
            .agg(['mean','sem']).reset_index())
rh.columns = ['model','memory','rh_mean','rh_sem']

# Panel D: gamma by model × memory (all sources combined)
gamma_rows = []
for (mdl, mem), grp in df.groupby(['model','memory'], observed=True):
    g = gamma_mod.goodman_kruskal_gamma(grp['accuracy'].values, grp['confidence'].values)
    n = len(grp)
    fz = np.arctanh(np.clip(g, -0.9999, 0.9999)) if not np.isnan(g) else np.nan
    gamma_rows.append({'model': mdl, 'memory': mem, 'gamma': g, 'f_gamma': fz, 'n': n})
df_gamma = pd.DataFrame(gamma_rows)

# Also gamma by model × memory × source (for supplemental)
gamma_src_rows = []
for (mdl, mem, src), grp in df.groupby(['model','memory','source'], observed=True):
    g = gamma_mod.goodman_kruskal_gamma(grp['accuracy'].values, grp['confidence'].values)
    fz = np.arctanh(np.clip(g, -0.9999, 0.9999)) if not np.isnan(g) else np.nan
    gamma_src_rows.append({'model':mdl,'memory':mem,'source':src,'gamma':g,'f_gamma':fz})
df_gamma_src = pd.DataFrame(gamma_src_rows)

print('Summaries computed.')
print(df_gamma[['model','memory','gamma','f_gamma']].to_string(index=False))

Summaries computed.
         model     memory     gamma   f_gamma
    Gemma3:12b SingleTurn  0.768694  1.017129
    Gemma3:12b TrialChain -0.282759 -0.290678
Gemma3:12b-QAT SingleTurn  0.428571  0.458145
Gemma3:12b-QAT TrialChain -0.011952 -0.011953
    Gemma3:27b SingleTurn       NaN       NaN
    Gemma3:27b TrialChain       NaN       NaN
Gemma3:27b-QAT SingleTurn       NaN       NaN
Gemma3:27b-QAT TrialChain       NaN       NaN
  Llama3.3:70b SingleTurn  0.976150  2.208559
  Llama3.3:70b TrialChain       NaN       NaN
 Llama4:16x17b SingleTurn  0.263158  0.269498
 Llama4:16x17b TrialChain -0.548260 -0.615890


In [4]:
# ── 3. Fig 3 — 4-panel manuscript figure ─────────────────────────────────
#
# Layout (2×2):
#  (a) Accuracy — SingleTurn, by model × source
#  (b) Accuracy — TrialChain, by model × source
#  (c) Reading hallucination (perceived/external), by model × memory
#  (d) Metacognitive gamma (Fisher's Z), by model × memory

MEMORIES = ['SingleTurn', 'TrialChain']
SOURCES  = ['imagined', 'perceived']
x        = np.arange(len(MODEL_ORDER))
bw       = 0.38

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig,
                        hspace=0.55, wspace=0.38,
                        left=0.08, right=0.97, top=0.93, bottom=0.13)

# ── Helper: accuracy grouped-bar panel ────────────────────────────────────
def _acc_panel(ax, memory, legend=False):
    sub = acc[acc['memory'] == memory]
    offsets = [-bw/2, bw/2]
    for i, src in enumerate(SOURCES):
        s = sub[sub['source'] == src].set_index('model').reindex(MODEL_ORDER)
        bars = ax.bar(x + offsets[i], s['acc_mean'].values, width=bw,
                      color=SRC_PAL[src], hatch=SRC_HATCH[src],
                      label=SRC_LABELS[src], alpha=0.88,
                      edgecolor='white', linewidth=0.5)
        ax.errorbar(x + offsets[i], s['acc_mean'].values,
                    yerr=s['acc_sem'].values,
                    fmt='none', ecolor='#333333', elinewidth=1.0, capsize=2.5)
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, fontsize=9.5, fontweight='bold',
                       rotation=40, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.65, len(MODEL_ORDER) - 0.35)
    ax.set_ylim(0, 1.09)
    ax.set_ylabel('Accuracy (Mean ± SEM)', fontweight='bold')
    ax.axhline(0.5, color='grey', lw=0.8, ls='--', alpha=0.5, zorder=0)
    if legend:
        ax.legend(title='Source', loc='lower right', framealpha=0.9)

# Panel A — SingleTurn
ax_a = fig.add_subplot(gs[0, 0])
_acc_panel(ax_a, 'SingleTurn', legend=True)
_panel_tag(ax_a, 'a', 'Accuracy — Single-Turn')

# Panel B — TrialChain
ax_b = fig.add_subplot(gs[0, 1])
_acc_panel(ax_b, 'TrialChain', legend=False)
_panel_tag(ax_b, 'b', 'Accuracy — Trial-Chain')

# Panel C — Reading Hallucination (horizontal bars)
ax_c = fig.add_subplot(gs[1, 0])
y_pos = np.arange(len(MODEL_ORDER))
bw_h  = 0.35
for i, mem in enumerate(MEMORIES):
    sub = rh[rh['memory'] == mem].set_index('model').reindex(MODEL_ORDER)
    ax_c.barh(y_pos + (0.5-i)*bw_h, sub['rh_mean'].values, height=bw_h,
              color=MEM_PAL[mem], label=MEM_LABELS[mem], alpha=0.88,
              edgecolor='white', linewidth=0.5)
    ax_c.errorbar(sub['rh_mean'].values, y_pos + (0.5-i)*bw_h,
                  xerr=sub['rh_sem'].values,
                  fmt='none', ecolor='#333333', elinewidth=1.0, capsize=2.5)
ax_c.set_yticks(y_pos)
ax_c.set_yticklabels(MODEL_LABELS, fontsize=9.5, fontweight='bold')
ax_c.set_xlim(0, 1.05)
ax_c.set_xlabel('Reading Hallucination Rate (Mean ± SEM)', fontweight='bold')
ax_c.axvline(0, color='black', lw=0.8)
ax_c.legend(title='Memory', loc='lower right', framealpha=0.9)
_panel_tag(ax_c, 'c', 'Reading Hallucinations (External/Perceived)')

# Panel D — Gamma (Fisher's Z)
ax_d = fig.add_subplot(gs[1, 1])
for i, mem in enumerate(MEMORIES):
    sub = df_gamma[df_gamma['memory'] == mem].set_index('model').reindex(MODEL_ORDER)
    ax_d.bar(x + (0.5-i)*bw, sub['f_gamma'].values, width=bw,
             color=MEM_PAL[mem], label=MEM_LABELS[mem], alpha=0.88,
             edgecolor='white', linewidth=0.5)
ax_d.axhline(0, color='black', lw=0.9, ls='--', alpha=0.6)
ax_d.set_xticks(x)
ax_d.set_xticklabels(MODEL_LABELS, fontsize=9.5, fontweight='bold',
                     rotation=40, ha='right', rotation_mode='anchor')
ax_d.set_xlim(-0.65, len(MODEL_ORDER) - 0.35)
ax_d.set_ylabel("Metacognitive Sensitivity\n(γ Fisher's Z)", fontweight='bold')
ax_d.legend(title='Memory', loc='upper left', framealpha=0.9)
_panel_tag(ax_d, 'd', 'Metacognitive Sensitivity')


for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(MAN / f'Fig3_exp1_key_results.{fmt}',
                dpi=dpi, bbox_inches='tight')
print('Fig 3 saved to', MAN)
plt.close('all')

Fig 3 saved to <PROJECT_ROOT>/reports/figures/manuscript


In [5]:
# ── 4. Supp Fig S1 — Confidence distributions ────────────────────────────
conf_levels = sorted(df['confidence'].dropna().astype(int).unique())
SOURCES2 = ['perceived', 'imagined']
COLS_MDL = plt.cm.tab10.colors

fig, axes = plt.subplots(2, 2, figsize=(14, 9),
                         sharex=True, sharey=True)

for row, mem in enumerate(MEMORIES):
    for col, src in enumerate(SOURCES2):
        ax = axes[row, col]
        sub = df[(df['memory'] == mem) & (df['source'] == src)]
        for mi, mdl in enumerate(MODEL_ORDER):
            grp = sub[sub['model'] == mdl]
            if grp.empty:
                continue
            p = (grp['confidence'].dropna().astype(int)
                 .value_counts(normalize=True)
                 .reindex(conf_levels, fill_value=0.0)
                 .sort_index())
            ax.plot(p.index, p.values, 'o-',
                    color=COLS_MDL[mi], lw=1.6, ms=5, alpha=0.85,
                    label=MODEL_LABELS[mi].replace('\n',' '))
        ax.set_title(f'{mem} · {src.capitalize()}',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Confidence Level')
        ax.set_ylabel('Proportion of Trials')
        ax.set_xticks(conf_levels)
        if row == 0 and col == 0:
            ax.legend(fontsize=8, ncol=2, loc='upper left')

plt.tight_layout()
for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(SUP / f'FigS1_exp1_confidence.{fmt}', dpi=dpi, bbox_inches='tight')
print('FigS1 saved')
plt.close('all')

FigS1 saved


In [6]:
# ── 5. Supp Fig S2 — Relatedness ratings ─────────────────────────────────
rr = (df.groupby(['model','memory','source'], observed=True)['rating']
        .agg(['mean','sem']).reset_index())
rr.columns = ['model','memory','source','rr_mean','rr_sem']

SRC_LS  = {'perceived': '-', 'imagined': '--'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for col, mem in enumerate(MEMORIES):
    ax = axes[col]
    sub = rr[rr['memory'] == mem]
    for src in SOURCES2:
        s = sub[sub['source'] == src].set_index('model').reindex(MODEL_ORDER)
        ax.errorbar(x, s['rr_mean'].values, yerr=s['rr_sem'].values,
                    color=SRC_PAL[src], ls=SRC_LS[src],
                    lw=2.0, marker='o', ms=6, capsize=3, alpha=0.9,
                    label=src.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, fontsize=9.5, fontweight='bold',
                       rotation=40, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.5, len(MODEL_ORDER) - 0.5)
    ax.set_ylabel('Relatedness Rating (Mean ± SEM)')
    ax.set_title(mem, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.legend(title='Source')

plt.tight_layout()
for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(SUP / f'FigS2_exp1_relatedness.{fmt}', dpi=dpi, bbox_inches='tight')
print('FigS2 saved')
plt.close('all')

FigS2 saved


In [7]:
# ── 6. Supp Fig S3 — Gamma z by model × memory × source ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for col, mem in enumerate(MEMORIES):
    ax = axes[col]
    sub = df_gamma_src[df_gamma_src['memory'] == mem]
    for i, src in enumerate(SOURCES2):
        s = sub[sub['source'] == src].set_index('model').reindex(MODEL_ORDER)
        ax.bar(x + (0.5-i)*bw, s['f_gamma'].values, width=bw,
               color=SRC_PAL[src], label=src.capitalize(),
               alpha=0.88, edgecolor='white', lw=0.5)
    ax.axhline(0, color='black', lw=0.9, ls='--', alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, fontsize=9.5, fontweight='bold',
                       rotation=40, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.65, len(MODEL_ORDER) - 0.35)
    ax.set_ylabel("γ Fisher's Z")
    ax.set_title(mem, fontsize=11, fontweight='bold')
    ax.legend(title='Source')

plt.tight_layout()
for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(SUP / f'FigS3_exp1_gamma_by_source.{fmt}', dpi=dpi, bbox_inches='tight')
print('FigS3 saved')
plt.close('all')

FigS3 saved


In [8]:
# ── 7. Summary ────────────────────────────────────────────────────────────
print('\n=== Exp 1 Figure Notebook Complete ===')
print('Manuscript:', list(MAN.glob('Fig3*')))
print('Supplemental:', list(SUP.glob('FigS[123]*exp1*')))


=== Exp 1 Figure Notebook Complete ===
Manuscript: [PosixPath('<PROJECT_ROOT>/reports/figures/manuscript/Fig3_exp1_key_results.pdf'), PosixPath('<PROJECT_ROOT>/reports/figures/manuscript/Fig3_exp1_key_results.png')]
Supplemental: [PosixPath('<PROJECT_ROOT>/reports/figures/supplemental/FigS1_exp1_confidence.png'), PosixPath('<PROJECT_ROOT>/reports/figures/supplemental/FigS1_exp1_confidence.pdf'), PosixPath('<PROJECT_ROOT>/reports/figures/supplemental/FigS3_exp1_gamma_by_source.png'), PosixPath('<PROJECT_ROOT>/reports/figures/supplemental/FigS3_exp1_gamma_by_source.pdf'), PosixPath('<PROJECT_ROOT>/reports/figures/supplemental/FigS2_exp1_relatedness.pdf'), PosixPath('<PROJECT_ROOT>/reports/figures/supplemental/FigS2_exp1_relatedness.png')]
